In [0]:
%pip install unidecode

from unidecode import unidecode
import unicodedata 

def _strip_accents(s):
    """à -> a, è -> e (tivu.tv usa diacritici, Auditel li perde)."""
    return "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))

# Suffissi editoriali — port da competitor_features._SUFFIXES (in minuscolo)
# + i pattern emersi nel debug del job 11.
_TITLE_SUFFIXES = [
    # Edizioni TG / pagine
    r"\s+edizione\s+straordinaria$", r"\s+ed\s+straordinaria$",
    r"\s+prima\s+pagina$", r"\s+breaking\s+news$",
    r"\s+ultim[ae]?\s+ora(\s+\w+)?$",
    r"\s+ore\s+\d{1,2}(\s+\d{1,2})?$",
    # Stagioni
    r"\s+prima\s+stagione.*$", r"\s+seconda\s+stagione.*$",
    r"\s+terza\s+stagione.*$", r"\s+quarta\s+stagione.*$",
    r"\s+stagione\s+\d+.*$",
    # Cicli stagionali / weekend
    r"\s+il\s+weekend.*$", r"\s+weekend.*$",
    r"\s+estate$", r"\s+cronache\s+d\s+estate$",
    r"\s+sabato$", r"\s+domenica$", r"\s+di\s+piu$",
    # Speciali / varianti
    r"\s+speciale.*$",
    r"\s+1\^?\s*visione$", r"\s+prima\s+visione$",
    r"\s+inizia\s+la\s+sfida.*$", r"\s+le\s+\d+\s+botole.*$",
    r"\s+prima\s+sfida$", r"\s+il\s+torneo\s+dei\s+campioni$",
    r"\s+il\s+torneo$", r"\s+cosa\s+vi\s+siete\s+persi$",
    # Eventi politici / cronaca
    r"\s+elezioni.*$", r"\s+referendum$", r"\s+si\s+no$", r"\s+si\s+o\s+no$",
    r"\s+il\s+bis\s+di\s+trump.*$", r"\s+il\s+ritorno\s+di\s+trump.*$",
    r"\s+la\s+morte\s+del\s+papa$",
    r"\s+diario\s+del\s+giorno.*$", r"\s+diario\s+della.*$",
    # Closing / segment markers
    r"\s+highlights$", r"\s+aftershow$", r"\s+after\s+show$",
    r"\s+buonanotte$", r"\s+saluti$", r"\s+i\s+saluti$",
    r"\s+rewind$", r"\s+compilation.*$", r"\s+tra\s+poco$",
    # Eventi sportivi numerati (Giro d'Italia 109^ edizione 12^ tap)
    r"\s+\d+\s*edizione.*$", r"\s+\d+\s*tap.*$",
    # Anno trailing (Giro d'Italia 2026 -> giro ditalia)
    r"\s+\d{4}$",
]

# Prefissi editoriali — port da get_family_key
_TITLE_PREFIXES = [
    r"^pres\.?\s*",
    r"^anteprima\s+",
    r"^ant\.?\s*",
    r"^i\s+saluti\s+di\s+",
    r"^la\s+buonanotte\s+di\s+",
]

def normalize_title(s):
    """Normalizzazione aggressiva applicata sia a palinsesto tivu.tv che ad hist Auditel.
    Port di `get_family_key` (competitor_features.py) + adattamenti per il debug 11_job_forecast.
    """
    if s is None: return ""
    s = str(s).lower().strip()
    s = _strip_accents(s)

    # 1. Parens: qualsiasi contenuto (lettere, anni, "Diretta", marker Auditel)
    s = re.sub(r"\s*\([^)]*\)\s*", " ", s)

    # 2. Prefissi
    for prefix in _TITLE_PREFIXES:
        s = re.sub(prefix, "", s)

    # 3. Pattern episodi numerati "X - X, N"
    s = re.sub(r"\s+-\s+.+,\s*\d+$", "", s)

    # 4. Abbreviazioni puntate (R.I.S. -> ris) - prima della punteggiatura
    s = re.sub(r"\b([a-z])(\.\s*[a-z])+\.?\b",
               lambda m: m.group(0).replace(".", "").replace(" ", ""), s)

    # 5. Apostrofi -> rimuovi (camera cafe' -> camera cafe, l'eredita -> leredita)
    s = re.sub(r"['\u2019\u2018`]", "", s)

    # 6. Punteggiatura residua -> spazio
    s = re.sub(r"[^a-z0-9 ]", " ", s)

    # 7. Suffissi editoriali (post-punteggiatura per non confondersi con `,`, `-` ecc.)
    for suffix in _TITLE_SUFFIXES:
        s = re.sub(suffix, "", s)

    # 8. Collapse spazi
    return re.sub(r"\s+", " ", s).strip()


MAPPING_MANUALE = {
    # ── Esistenti validi ──
    ("Rai 1", "telegiornale"):              "tg1",
    ("Rai 1", "che tempo fa"):              "meteo 1",
    ("Canale 5", "meteo"):                  "il meteo",
    ("Rete 4", "tg4 telegiornale"):         "tg4",
    ("Italia 1", "macgyver"):               "mac gyver",

    # ── Nuovi (target verificati in hist) ──
    # TG1 con/senza spazio (4371 righe storiche disponibili)
    ("Rai 1", "tg 1"):                      "tg1",
    # RaiNews24: hist usa "rai news" (1660 Rai1, 2308 Rai3)
    ("Rai 1", "rainews24"):                 "rai news",
    ("Rai 3", "rainews24"):                 "rai news",
    # TGR Rai 3 (943-996 righe)
    ("Rai 3", "piazza affari"):             "tgr piazza affari",
    ("Rai 3", "tg regione meteo"):          "tgr meteo",
    # Sky TG24 Buongiorno (137 righe in hist con prefisso "sky")
    ("Tv8", "tg24 buongiorno"):             "sky tg24 buongiorno",
    # Cifra in lettere (solo 2 righe storiche → match flebile ma onesto)
    ("Rete 4", "un esercito di 5 uomini"):  "un esercito di cinque uomini",
}

def apply_manual_mapping(canale, prog_norm):
    return MAPPING_MANUALE.get((canale, prog_norm), prog_norm)